In [240]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler



df= pd.read_csv('../data collections/WineQT (1).csv')

In [241]:
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,0
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,1
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,2
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,3
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1138,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6,1592
1139,6.8,0.620,0.08,1.9,0.068,28.0,38.0,0.99651,3.42,0.82,9.5,6,1593
1140,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5,1594
1141,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6,1595


In [242]:

print(df.nunique())
print(df.info())
print(df.isna().sum())
df.describe()

fixed acidity             91
volatile acidity         135
citric acid               77
residual sugar            80
chlorides                131
free sulfur dioxide       53
total sulfur dioxide     138
density                  388
pH                        87
sulphates                 89
alcohol                   61
quality                    6
Id                      1143
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8  

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
count,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000
mean,8.311111,0.531339,0.268364,2.532152,0.086933,15.615486,45.914698,0.996730,3.311015,0.657708,10.442111,5.657043,804.969379
std,1.747595,0.179633,0.196686,1.355917,0.047267,10.250486,32.782130,0.001925,0.156664,0.170399,1.082196,0.805824,463.997116
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000,0.000000
25%,7.100000,0.392500,0.090000,1.900000,0.070000,7.000000,21.000000,0.995570,3.205000,0.550000,9.500000,5.000000,411.000000
50%,7.900000,0.520000,0.250000,2.200000,0.079000,13.000000,37.000000,0.996680,3.310000,0.620000,10.200000,6.000000,794.000000
75%,9.100000,0.640000,0.420000,2.600000,0.090000,21.000000,61.000000,0.997845,3.400000,0.730000,11.100000,6.000000,1209.500000
max,15.900000,1.580000,1.000000,15.500000,0.611000,68.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000,1597.000000


In [243]:
X= df.drop(['quality', 'Id'], axis=1)
y=df.quality


X_train, X_test, y_train, y_test= train_test_split(X, y, stratify=y, random_state=0)

svc_linear= SVC(kernel='linear', random_state=0)
svc_rbf= SVC(kernel='rbf', random_state=0)
lr= LogisticRegression(max_iter=10000, random_state=0)
rf= RandomForestClassifier(random_state=0)
dt= DecisionTreeClassifier(random_state=0)

In [244]:
X_train_capped = X_train.copy()
X_test_capped = X_test.copy()

cols= X_train.columns

for col in cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    X_train_capped[col] = X_train_capped[col].clip(lower, upper)
    X_test_capped[col] = X_test_capped[col].clip(lower, upper)



In [245]:
scaler= StandardScaler()

X_train_scaled= scaler.fit_transform(X_train)
X_test_scaled= scaler.transform(X_test)

X_train_clean= scaler.fit_transform(X_train_capped)
X_test_clean= scaler.transform(X_test_capped)


In [246]:
datasets= {
    'Raw': (X_train, X_test),
    'Scaled': (X_train_scaled, X_test_scaled),
    'Scaled and clipped': (X_train_clean, X_test_clean)
}

models= {
    'LogisticRegression': LogisticRegression(max_iter=10000, random_state=0),
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=0),
    'RandomForestClassifier': RandomForestClassifier(random_state=0),
    'SVC Linear': SVC(kernel='linear', random_state=0),
    'SVC RBF': SVC(kernel='rbf', random_state=0)
}


results= []

for data, (xtrain, xtest) in datasets.items():
    for name, model in models.items():
       
        model.fit(xtrain, y_train)

        train_score= model.score(xtrain, y_train)
        test_score= model.score(xtest, y_test)

        results.append(
            {
                'data': data,
                'model': name,
                'trainscore': train_score,
                'testscore': test_score
            }
        )

results= pd.DataFrame(results).sort_values('testscore', ascending=False)
results

,data,model,trainscore,testscore
12,Scaled and clipped,RandomForestClassifier,1.000000,0.646853
7,Scaled,RandomForestClassifier,1.000000,0.636364
2,Raw,RandomForestClassifier,1.000000,0.632867
9,Scaled,SVC RBF,0.686114,0.615385
10,Scaled and clipped,LogisticRegression,0.620770,0.615385
14,Scaled and clipped,SVC RBF,0.702450,0.608392
8,Scaled,SVC Linear,0.606768,0.601399
13,Scaled and clipped,SVC Linear,0.612602,0.597902
0,Raw,LogisticRegression,0.621937,0.594406
5,Scaled,LogisticRegression,0.634772,0.590909


In [ ]:
rf= RandomForestClassifier(random_state=0, n_jobs=2)

params={
    'n_estimators': [200, 400],
    'max_depth': [4, 8],
    'min_samples_split': [4, 6],
    'min_samples_leaf': [4, 6, 10],
    'max_features':['sqrt', 'log2']
}

grid_rf= GridSearchCV(rf, params, cv=5, scoring='accuracy', n_jobs=4)

grid_rf.fit(X_train_clean, y_train)

print("Best params:", grid_rf.best_params_)
print("Best CV score:", grid_rf.best_score_)

print('Train:', grid_rf.best_estimator_.score(X_train_clean, y_train))
print('Test:', grid_rf.best_estimator_.score(X_test_clean, y_test))



Best params: {'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 4, 'n_estimators': 200}
Best CV score: 0.6243234054127567
Train: 0.8109684947491248
Test: 0.6083916083916084


In [267]:
svc= SVC(random_state=0, kernel='rbf')

params={
    'C': [0.1, 1, 10, 100],
    'gamma': ["scale", 0.01, 0.1, 1]
}

grid_svc= GridSearchCV(svc, params, cv=5, scoring='f1_weighted', n_jobs=4)

grid_svc.fit(X_train_scaled, y_train)

print("Best params:", grid_svc.best_params_)
print("Best CV score:", grid_svc.best_score_)

print('Train:', grid_svc.best_estimator_.score(X_train_scaled, y_train))
print('Test:', grid_svc.best_estimator_.score(X_test_scaled, y_test))


for c in [0.1, 1, 10, 100]:
    for g in ["scale", 0.01, 0.1, 1]:
        svc2= SVC(random_state=0, kernel='rbf', C=c, gamma=g)
        svc2.fit(X_train_scaled, y_train)

        print(f'\ntrain: {svc2.score(X_train_scaled, y_train)} C: {c} G: {g}')
        print(f'test: {svc2.score(X_test_scaled, y_test)}')

#C= 0.1 G= 0.1 0.60 > 0.59
#C=1 G='scale' 0.68 > 0.61

Best params: {'C': 10, 'gamma': 0.1}
Best CV score: 0.6197554287841758
Train: 0.8214702450408401
Test: 0.6223776223776224

train: 0.6021003500583431 C: 0.1 G: scale
test: 0.5909090909090909

train: 0.5705950991831972 C: 0.1 G: 0.01
test: 0.5559440559440559

train: 0.6056009334889149 C: 0.1 G: 0.1
test: 0.5909090909090909

train: 0.42473745624270715 C: 0.1 G: 1
test: 0.4230769230769231

train: 0.6861143523920653 C: 1 G: scale
test: 0.6153846153846154

train: 0.5950991831971996 C: 1 G: 0.01
test: 0.583916083916084

train: 0.6977829638273045 C: 1 G: 0.1
test: 0.6118881118881119

train: 0.9568261376896149 C: 1 G: 1
test: 0.6118881118881119

train: 0.8074679113185531 C: 10 G: scale
test: 0.6153846153846154

train: 0.6487747957992999 C: 10 G: 0.01
test: 0.6013986013986014

train: 0.8214702450408401 C: 10 G: 0.1
test: 0.6223776223776224

train: 0.9976662777129521 C: 10 G: 1
test: 0.6013986013986014

train: 0.9381563593932322 C: 100 G: scale
test: 0.5384615384615384

train: 0.691948658109685 C

In [266]:
lr= LogisticRegression(random_state=0, max_iter=10000)

params={
    'C':[0.001, 0.01, 0.1, 1, 10,],
    'class_weight': [None, 'balanced']
}

grid_lr = GridSearchCV(
    lr,
    params,
    cv=5,
    scoring='accuracy',
    n_jobs=4
)

grid_lr.fit(X_train_clean, y_train)

best_lr = grid_lr.best_estimator_

print('Best params:', grid_lr.best_params_)
print('Best CV score:', grid_lr.best_score_)

print('Train:', best_lr.score(X_train_scaled, y_train))
print('Test:', best_lr.score(X_test_scaled, y_test))




Best params: {'C': 0.1, 'class_weight': None}
Best CV score: 0.603250373997008
Train: 0.6207701283547258
Test: 0.6048951048951049
